# MedStream - Data Cleaning & Preprocessing

This notebook processes the raw diabetes prediction dataset:
- Inspects shape and missing values
- Drops duplicate patient records
- Filters out rare/invalid categories
- Groups smoking status into cleaner categories
- Encodes categorical fields and exports to `data/processed/diabetes_cleaned.csv`

### 1. Load Data

In [ ]:
import pandas as pd
import numpy as np

raw_path = "data/raw/diabetes_prediction_dataset.csv"
df = pd.read_csv(raw_path)
print("Initial shape:", df.shape)
df.head()

### 2. Check Data Types & Nulls

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

### 3. Remove Duplicate Rows

In [ ]:
duplicates = df.duplicated().sum()
print(f"Found {duplicates:,} duplicate rows")

df = df.drop_duplicates()
print(f"Remaining rows after drop: {len(df):,}")

### 4. Filter Gender Values

In [ ]:
print("Gender counts:")
print(df['gender'].value_counts())

# keep binary gender labels (dropping 18 'Other' rows)
df = df[df['gender'].isin(['Female', 'Male'])]
print("\nFiltered gender counts:")
print(df['gender'].value_counts())

### 5. Simplify Smoking History
Map varied statuses into 4 main groups: `never`, `former`, `current`, and `No Info`.

In [ ]:
def map_smoking(val):
    if val in ['former', 'not current', 'ever']:
        return 'former'
    elif val == 'current':
        return 'current'
    elif val == 'never':
        return 'never'
    return 'No Info'

df['smoking_history'] = df['smoking_history'].apply(map_smoking)
print(df['smoking_history'].value_counts())

### 6. Encode Categories and Types

In [ ]:
# convert gender to numeric flag (0 = Female, 1 = Male)
df['gender_code'] = df['gender'].map({'Female': 0, 'Male': 1})

# cast age float to integer
df['age'] = df['age'].astype(int)

df.head()

### 7. Target Distribution Check

In [ ]:
counts = df['diabetes'].value_counts()
pct = (df['diabetes'].value_counts(normalize=True) * 100).round(2)
target_summary = pd.DataFrame({'Count': counts, 'Percentage (%)': pct})
target_summary.index = ['Negative (0)', 'Positive (1)']
target_summary

### 8. Export Cleaned Dataset

In [ ]:
import os

out_dir = "data/processed"
os.makedirs(out_dir, exist_ok=True)

out_file = f"{out_dir}/diabetes_cleaned.csv"
df.to_csv(out_file, index=False)
print(f"Saved {len(df):,} cleaned rows to {out_file}")